# Magic Card To Text Dataset Generator  
__Objective:__ The aim of this notebook is to harness the IBM Granite Docling model to convert a large batch of Magic the Gathering card images into a structured text dataset. That dataset can then be used to fine tune a transformer model on multi-label classification of scryfall tags.

## Packages and Data

In [ ]:
# packages

## link directory
from pathlib import Path
import sys

workspace_root = Path.cwd()
if workspace_root.name == 'notebooks':
    workspace_root = workspace_root.parent
if str(workspace_root) not in sys.path:
    sys.path.append(str(workspace_root))

## custom packages
from src.card_ocr_v1.dataset import load_manifest_records, summarize_manifest

In [2]:
# retrieve core data
manifest_path = workspace_root / "data" / "card_image_text_manifest.jsonl"
records = load_manifest_records(manifest_path, skip_missing_images=True)
summary = summarize_manifest(records)
summary

{'num_records': 8443, 'split_counts': {'train': 6713, 'val': 1680, 'test': 50}}

In [10]:
train_records = [record for record in records if record['split'] == 'train']
val_records = [record for record in records if record['split'] == 'val']
test_records = [record for record in records if record['split'] == 'test']
print(f'Train: {len(train_records)} | Test: {len(test_records)} | Val: {len(val_records)}')

Train: 6713 | Test: 50 | Val: 1680


In [ ]:
from docling.document_converter import DocumentConverter
from docling.datamodel.pipeline_options import (
    PdfPipelineOptions,
    VlmPipelineOptions
)
from docling.datamodel.base_models import InputFormat
from docling.document_converter import ImageFormatOption

from docling.document_converter import DocumentConverter
from docling.datamodel.pipeline_options import (
    PdfPipelineOptions,
    VlmPipelineOptions
)
from docling.datamodel.base_models import InputFormat
from docling.document_converter import ImageFormatOption

from tqdm import tqdm

class DoclingCardTextDatasetGenerator():
    """
    Description
    ----------
    This class contains the necessary methods to convert a list of card image 
    paths into extracted plain text, and store the resulting file as a 
    new list of dicts (json-like object)

    NOTE: This class assumes we want to translate card images to text *one at a 
    time*. 

    Inputs
    ----------
    data = A list of dicts, where each dict is a collection of infomrmation about
        a card, critically including image_path
    use_granite = If true, we use the granite_docling version of the model instead
    """
    def __init__(
        self, 
        data:list[dict],
        use_granite:bool = False
    ):
        super().__init__()

        # store params as objects
        self.data = data 

        # instantiate objects for later use
        self.generated_text = [] # to be populated with dicts

        # instantiate the converter
        if use_granite:
            # configure VLM pipeline for granite
            pipeline_options = VlmPipelineOptions()
            pipeline_options.vlm_model = 'granite_docling'
            
            # set up the converter
            self.converter = DocumentConverter(
                format_options = {
                    InputFormat.IMAGE: ImageFormatOption(
                        pipeline_options = pipeline_options
                    )
                }
            )
        else:
            # accept the default options
            self.converter = DocumentConverter()

    # === Main Methods ===

    def run(self):
        """
        Description
        ----------
        This public method iterates through each record in `self.data` to invoke
        the converter and generate the final output of text.

        Inputs
        ----------

        Returns
        ----------
        None, but self.generated_text will be populated in a similar structure to 
        self.data, except we have 'card_text_ocr' rather than Scryfall cleaned card text.
        """
        # define the progress bar
        # progress_bar = tqdm(len(self.data), desc = 'Reading card text from card images')
        # for record in self.data:
        for record in tqdm(self.data, desc = 'Reading card text from card images'):
            # invoke the converter
            gen_text = self._read_image(record = record)

            # define output
            out = {
                'id': record['id'],
                'oracle_id': record['oracle_id'],
                'card_name': record['card_name'],
                'card_text_ocr': gen_text,
                'tags': record['tags']
            }

            # store output
            self.generated_text.append(out)

    # === Internal Methods ===
    
    def _read_image(self, record:dict):
        """
        Description
        ----------
        This internal method is what gets iteratively called while looping through
        the `self.data` object to generate the requisite text.

        Inputs
        ----------
        record = The record from which we want to extract text

        Returns
        ----------

        """
        # define card path
        image_path = Path(record['image_path'])

        # run inference
        result = self.converter.convert(image_path)

        return result.document.export_to_text()


dataset_generator = DoclingCardTextDatasetGenerator(
    data = test_records,
    use_granite = False
)
dataset_generator.run()

Reading card text from card images:   0%|          | 0/50 [00:00<?, ?it/s][INFO] 2026-07-20 12:28:08,229 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-07-20 12:28:08,229 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-07-20 12:28:08,238 [RapidOCR] download_file.py:60: File exists and is valid: /Users/nickcruickshank/Projects/mtg-multimodal-classification/.venv/lib/python3.14/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-07-20 12:28:08,238 [RapidOCR] main.py:50: Using /Users/nickcruickshank/Projects/mtg-multimodal-classification/.venv/lib/python3.14/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-07-20 12:28:08,323 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-07-20 12:28:08,323 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-07-20 12:28:08,325 [RapidOCR] download_file.py:60: File exists and is valid: /Users/nickcruickshank/Projects/mtg-multimodal-classification/.venv/lib/python3.14/site-pa

In [23]:
dataset_generator.generated_text

[{'id': 538,
  'oracle_id': '044a169e-32ae-464a-a642-8d3409e82d9f',
  'card_name': 'Flailing Manticore',
  'card_text_ocr': 'Flailing Manticore\n\nCreature --- Monster\n\nFlying, first strike\n\n- 1: Flailing Manticore gets +1/+1 until end of turn. Any player may play this ability.\n- 1: Flailing Manticore gets -1/-1 until end of turn. Any player may play this ability.',
  'tags': ['activated ability',
   'bottomless mana sink',
   'drawback',
   'evasion',
   'shade pump',
   'type errata']},
 {'id': 573,
  'oracle_id': '047e761e-7bbe-403b-86d7-cb9fb0b20b21',
  'card_name': 'Plague Stinger',
  'card_text_ocr': 'Plague Stinger\n\nCreature -- Insect Horror\n\nFlying\n\nInfect (This creature deals damage to creatures in the form of -1/-1 counters and to players in the form of poison counters.)\n\nIt leaves its victims one sting closer to phyresis.',
  'tags': ['evasion',
   'french vanilla',
   'gives mm counters',
   'poisonous',
   'type addition phyrexian']},
 {'id': 722,
  'oracle_id

In [22]:
for i, record in enumerate(dataset_generator.generated_text):
    if i >= 45:
        print(f'\n{"-" * 10}\nCard = {record["card_name"]}\nGenerated Text:\n{record["card_text_ocr"]}\n{"-" * 10}\n')


----------
Card = Clockspinning
Generated Text:
Clockspinning

Instant

区

Buyback 3 (You may pay an additional 3 as you play this spell. If you do, put this card into your hand as it resolves.)

Choose a counter on target permanent or suspended card. Remove that counter from that permanent or card or put another of those counters on it.
----------


----------
Card = Vigean Hydropon
Generated Text:
Vigean Hydropon

Creature -- Plant Mutant

Graft 5 (This creature comes into play with five +1/+1 counters on it. Whenever another creature comes into play, you may move a +1/+1 counter from this creature onto it.)

Vigean Hydropon can't attack or block.

Fruits of magic, roots in science.
----------


----------
Card = Skirmish Rhino
Generated Text:
Skirmish Rhino

CreatureRhino

Trample

When this creature enters, each opponent loses 2 life and you 1 gain 2 life.

Slowly, the clans have 2 begun to recover and adapt long-forgotten tactics.

3/4
----------


----------
Card = Exalted Sunbo